# ENG50 — Numerical Integration Demo Notebook
**Introduction to Engineering · Module 5 — Integration in Python**

**Integration** answers the question: *how much total has accumulated?*

> Area under the curve of f(x) from a to b

$$\int_a^b f(x)\, dx$$

- The **function** f(x) tells us the rate at each moment (speed, power, flow rate…)
- The **integral** gives the total (distance, energy, volume…)

We use Python's `scipy.integrate.quad` and `numpy` to compute these numerically — no antiderivative by hand required.

In [ ]:
from scipy.integrate import quad, trapezoid
import numpy as np
import matplotlib.pyplot as plt
print("Libraries loaded!")

---
## Example 1: Distance from Velocity ⚡

A car accelerates. Its speed at time t is v(t) = 3t² m/s. How far does it travel from t = 0 to t = 4 seconds?

**Where the integral comes from — the analytic check:**

$$\int_0^4 3t^2 \, dt = \left[ t^3 \right]_0^4 = 4^3 - 0^3 = 64 \text{ meters}$$

**Key idea:** velocity × time = distance. If velocity is changing, we slice time into tiny pieces, multiply v × Δt for each slice, and add them up — that is integration.

```
total distance = sum of (velocity × tiny time step) for every step
              = ∫ v(t) dt
```

In [ ]:
# ── Step 1: Define the function to integrate ───────────
def velocity(t):
    return 3 * t**2    # v(t) = 3t²  [m/s]

# ── Step 2: Integrate with quad ────────────────────────
# quad(function, lower_limit, upper_limit)
# Returns two numbers: (result, estimated_error)
distance, error = quad(velocity, 0, 4)

print(f"Total distance: {distance:.4f} m")
print(f"Estimated error: {error:.2e}")   # e.g. 7.1e-13 means 0.0000000000007

# ── Step 3: Visualize — shade the area under the curve ─
t = np.linspace(0, 4, 300)
v = velocity(t)

plt.figure(figsize=(7, 4))
plt.plot(t, v, 'b-', linewidth=2, label='v(t) = 3t²')
plt.fill_between(t, v, alpha=0.3, color='blue', label=f'Area = {distance:.1f} m')
plt.xlabel('Time (s)')
plt.ylabel('Velocity (m/s)')
plt.title('Distance = Area Under Velocity Curve')
plt.grid(True)
plt.legend()
plt.show()

### 🔍 Deep Dive — How quad works internally

**The 3 arguments:**
```python
quad(velocity,  0,   4)
#    ^          ^    ^
#    function   a    b
```

**What quad returns:**
```python
distance, error = quad(velocity, 0, 4)
#  ^         ^
#  result    estimated error (how much it might be off)
```

**The Riemann sum idea (what integration really is):**

Split [0, 4] into N tiny slices of width Δt. For each slice, compute f(t) × Δt (a thin rectangle). Add all rectangles up:

$$\int_0^4 f(t)\,dt \approx \sum_{i=1}^{N} f(t_i) \cdot \Delta t$$

As Δt → 0, the approximation becomes exact. `quad` does this adaptively — it uses more slices where the curve bends sharply, fewer where it's flat, to stay accurate without wasting computation.

### ✏️ Your Turn — Example 1

1. Change the velocity to `v(t) = 10*t` (constant acceleration). What is the distance from t=0 to t=5?
2. Try `v(t) = np.sin(t)`. Integrate from 0 to π. What do you expect the answer to be?
3. Integrate `v(t) = 3*t**2` from t=0 to t=10. Does it equal 1000?

---
## Example 2: Electrical Energy (Power Over Time) ⚡🔋

A device draws power P(t) = 50·sin²(t) watts. How much energy (in joules) is consumed in the first 10 seconds?

**Where the integral comes from:**

$$\text{Energy} = \int_0^{10} P(t)\, dt = \int_0^{10} 50 \sin^2(t)\, dt$$

Using the identity sin²(t) = (1 − cos(2t))/2:

$$= 50 \int_0^{10} \frac{1 - \cos(2t)}{2}\, dt = 25\left[t - \frac{\sin(2t)}{2}\right]_0^{10} \approx 248.5 \text{ J}$$

**Key idea:** the power curve fluctuates — sometimes the device draws more, sometimes less. The integral adds up all those varying contributions to give the *total* energy.

In [ ]:
# ── Define power function ──────────────────────────────
def power(t):
    return 50 * np.sin(t)**2    # P(t) = 50·sin²(t) [Watts]

# ── Integrate: total energy from t=0 to t=10 ──────────
energy, error = quad(power, 0, 10)
print(f"Total energy consumed: {energy:.2f} J")
print(f"Estimated error:       {error:.2e}")

# ── Also compute: average power ───────────────────────
# Average = total energy / total time
avg_power = energy / 10
print(f"Average power:         {avg_power:.2f} W")

# ── Plot ──────────────────────────────────────────────
# t      → 1D array of times:   [0, 0.033, ..., 10]
# p      → 1D array of power:   [0, 0.054, ..., 2.3]  (Watts)
t = np.linspace(0, 10, 500)
p = power(t)

plt.figure(figsize=(8, 4))
plt.plot(t, p, 'r-', linewidth=2, label='P(t) = 50·sin²(t)')
plt.fill_between(t, p, alpha=0.25, color='red', label=f'Total energy = {energy:.1f} J')
plt.axhline(avg_power, color='navy', linestyle='--', label=f'Avg power = {avg_power:.1f} W')
plt.xlabel('Time (s)')
plt.ylabel('Power (W)')
plt.title('Electrical Energy = Area Under Power Curve')
plt.grid(True)
plt.legend()
plt.show()

### ✏️ Your Turn — Example 2

1. Change the power to `P(t) = 100` (constant 100 W). What is the energy in 10 seconds? Does your answer make sense?
2. Try `P(t) = t * np.exp(-0.2*t)` (power that rises then falls). Find the total energy from 0 to 20 s.
3. Modify the code to find the average power for your new function.

---
## Example 3: Water Tank Volume (Flow Rate) 💧

Water flows into a tank at a rate that changes over time:
$$Q(t) = 10 + 5\cos\!\left(\frac{\pi t}{6}\right) \text{ liters/hour}$$

How much total water enters the tank in the first 12 hours?

**Where the integral comes from:**

$$V = \int_0^{12} Q(t)\, dt = \int_0^{12} \left(10 + 5\cos\!\frac{\pi t}{6}\right) dt
= \left[10t + \frac{30}{\pi}\sin\!\frac{\pi t}{6}\right]_0^{12} = 120 \text{ L}$$

Notice the cosine term cancels out over a full cycle — only the constant 10 L/hr contributes to the net total. The integration reveals this automatically.

**This example also shows the Trapezoidal Rule** — a simple numerical method you can apply when you only have *data points* (like sensor readings) rather than a formula.

In [ ]:
# ── Method 1: quad (uses a formula) ───────────────────
def flow_rate(t):
    return 10 + 5 * np.cos(np.pi * t / 6)   # Q(t) [L/hr]

volume_exact, _ = quad(flow_rate, 0, 12)
print(f"Volume (quad):         {volume_exact:.4f} L")

# ── Method 2: Trapezoidal Rule (uses data points only) ─
# Imagine a sensor logging flow every hour — no formula needed
# t_data   → hourly time stamps:  [0, 1, 2, ..., 12]
# Q_data   → sensor readings:     [15.0, 14.1, 11.5, ..., 15.0] (L/hr)
t_data = np.linspace(0, 12, 13)          # one reading per hour
Q_data = flow_rate(t_data)               # simulate the sensor

volume_trap = trapezoid(Q_data, t_data)  # trapezoid rule on the data
print(f"Volume (trapezoid):    {volume_trap:.4f} L")
print(f"Difference:            {abs(volume_exact - volume_trap):.4f} L")

# ── Plot: compare both methods ─────────────────────────
t_fine = np.linspace(0, 12, 400)
Q_fine = flow_rate(t_fine)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: quad (smooth fill)
axes[0].plot(t_fine, Q_fine, 'b-', linewidth=2)
axes[0].fill_between(t_fine, Q_fine, alpha=0.3, color='blue')
axes[0].set_title(f'quad  →  {volume_exact:.1f} L')
axes[0].set_xlabel('Time (hr)'); axes[0].set_ylabel('Flow rate (L/hr)')
axes[0].grid(True)

# Right: trapezoidal (trapezoids visible)
axes[1].plot(t_fine, Q_fine, 'b-', linewidth=1, alpha=0.4, label='True curve')
axes[1].plot(t_data, Q_data, 'ro-', linewidth=2, label='Sensor readings')
axes[1].fill_between(t_data, Q_data, alpha=0.3, color='red')
axes[1].set_title(f'Trapezoid Rule  →  {volume_trap:.1f} L')
axes[1].set_xlabel('Time (hr)'); axes[1].set_ylabel('Flow rate (L/hr)')
axes[1].legend(); axes[1].grid(True)

plt.suptitle('Water Tank — Two Integration Methods', fontweight='bold')
plt.tight_layout()
plt.show()

### 🔍 Deep Dive — Trapezoidal Rule

Instead of rectangles, the **trapezoidal rule** connects neighboring data points with straight lines and sums the areas of the trapezoids formed:

$$\int_a^b f(x)\,dx \approx \sum_{i=0}^{N-1} \frac{f(x_i) + f(x_{i+1})}{2} \cdot \Delta x$$

**When to use which method:**

| Situation | Use |
|---|---|
| You have a formula for f(x) | `quad` — more accurate |
| You only have sensor/experimental data | `trapezoid` — works without a formula |
| You need very high accuracy | `quad` — adaptive, error-controlled |
| Data is evenly spaced | Either works; `trapezoid` is simpler |

**More data points → more accurate trapezoid.** Try changing `np.linspace(0, 12, 13)` to `np.linspace(0, 12, 100)` and see how much closer the two answers get.

### ✏️ Your Turn — Example 3

1. Change to 25 hourly readings (`np.linspace(0, 12, 25)`). Does the trapezoidal answer get closer to the exact value?
2. Change `Q(t) = 20` (constant flow). What is the total volume in 12 hours? Does integration give the same answer as 20 × 12?
3. Suppose a sensor records these hourly readings: `[8, 10, 13, 11, 9, 12, 14, 10, 8, 11, 13, 9, 10]`. Use `trapezoid` to find the total volume. No formula needed!

---
## Example 4: Probability (Area Under a Bell Curve) 📊

In a statistics class, test scores follow a normal distribution with mean μ = 75 and standard deviation σ = 10. What fraction of students score between 65 and 85?

**The normal distribution formula:**

$$f(x) = \frac{1}{\sigma\sqrt{2\pi}}\, e^{-\frac{(x-\mu)^2}{2\sigma^2}}$$

There is **no closed-form antiderivative** for this — you cannot integrate it by hand to get a simple formula. This is exactly where numerical integration shines.

$$P(65 \leq X \leq 85) = \int_{65}^{85} f(x)\, dx$$

**Key idea:** probability = area under the curve. The total area under a probability curve is always exactly 1 (100%).

In [ ]:
# ── Parameters ────────────────────────────────────────
mu    = 75    # mean score
sigma = 10    # standard deviation

# ── Normal distribution function ───────────────────────
def normal_pdf(x, mu, sigma):
    # pdf = probability density function
    coefficient = 1 / (sigma * np.sqrt(2 * np.pi))
    exponent    = -((x - mu)**2) / (2 * sigma**2)
    return coefficient * np.exp(exponent)

# ── quad with extra arguments ─────────────────────────
# args=(mu, sigma) passes mu and sigma into normal_pdf
prob_65_85, _  = quad(normal_pdf, 65, 85, args=(mu, sigma))
prob_total, _  = quad(normal_pdf, -np.inf, np.inf, args=(mu, sigma))  # should be 1.0

print(f"P(65 ≤ X ≤ 85) = {prob_65_85:.4f}  ({prob_65_85*100:.1f}%)")
print(f"Total area     = {prob_total:.6f}   (should be exactly 1.0)")

# ── Plot ──────────────────────────────────────────────
# x      → score values:   [40, 40.3, ..., 110]
# y      → density values: [0.0001, 0.0002, ..., 0.0001]
#           y is NOT probability — it is probability DENSITY
#           Probability = area (y × width of interval)
x     = np.linspace(40, 110, 500)
y     = normal_pdf(x, mu, sigma)
x_mid = np.linspace(65, 85, 300)
y_mid = normal_pdf(x_mid, mu, sigma)

plt.figure(figsize=(8, 4))
plt.plot(x, y, 'darkgreen', linewidth=2, label='Score distribution')
plt.fill_between(x_mid, y_mid, alpha=0.4, color='green',
                 label=f'P(65≤X≤85) = {prob_65_85*100:.1f}%')
plt.axvline(mu, color='black', linestyle='--', alpha=0.5, label=f'Mean = {mu}')
plt.xlabel('Test Score')
plt.ylabel('Probability Density')
plt.title('Probability = Area Under Normal Curve')
plt.grid(True)
plt.legend()
plt.show()

### 🔍 Deep Dive — Passing extra arguments to quad

When your function needs parameters beyond just x, use `args=`:

```python
quad(normal_pdf, 65, 85, args=(mu, sigma))
#                         ^
#                         tuple of extra args passed to normal_pdf
```

This calls `normal_pdf(x, mu, sigma)` for every x that quad evaluates.

**Integrating to ±infinity:**
```python
quad(f, -np.inf, np.inf)   # works! quad handles infinite limits
```

**The 68-95-99.7 rule — verify it with quad:**
- 68% of data falls within 1σ of the mean
- 95% within 2σ
- 99.7% within 3σ

Try integrating from `mu - sigma` to `mu + sigma` and see if you get ≈ 0.68.

### ✏️ Your Turn — Example 4

1. What percentage of students score above 90? Integrate from 90 to `np.inf`.
2. Change `sigma = 5` (tighter bell curve). Does more or less area fall between 65 and 85?
3. Verify the 68% rule: integrate from `mu - sigma` to `mu + sigma`. Do you get ≈ 0.6827?

---
## Common Mistakes

| Mistake | Fix |
|---|---|
| `quad(f(), a, b)` | No parentheses on the function: `quad(f, a, b)` |
| `result = quad(f, a, b)` | quad returns TWO values: `result, error = quad(f, a, b)` |
| `trapezoid(y)` | Must pass x too: `trapezoid(y, x)` |
| Forgetting `x` in function definition | Always `def f(x):` even with extra args |
| `args=mu` (single value) | Must be a tuple: `args=(mu,)` or `args=(mu, sigma)` |
| Integrating past your data range | `t_eval` and limits must match your data |

---
## Quick Reference Template

```python
from scipy.integrate import quad, trapezoid
import numpy as np

# ── Option A: You have a formula ──────────────────────
def my_function(x):
    return x**2 + 3*x    # replace with your formula

result, error = quad(my_function, 0, 10)   # integrate from 0 to 10
print(f"Integral = {result:.4f}  (error ≈ {error:.1e})")

# ── Option B: You only have data points ───────────────
x_data = np.array([0, 1, 2, 3, 4, 5])      # x values
y_data = np.array([0, 1, 4, 9, 16, 25])    # f(x) values

result_trap = trapezoid(y_data, x_data)     # trapezoidal rule
print(f"Integral ≈ {result_trap:.4f}")
```

**quad arguments:**

| argument | meaning |
|---|---|
| `my_function` | the function to integrate (no parentheses) |
| `0` | lower limit a |
| `10` | upper limit b |
| `args=(p1, p2)` | extra parameters to pass to the function (optional) |

**trapezoid arguments:**

| argument | meaning |
|---|---|
| `y_data` | the function values (heights of the trapezoids) |
| `x_data` | the x positions (widths come from spacing) |

---
## Final Challenge — Design Your Own

Using the template above, solve one of these:

1. **Braking distance:** A car brakes with deceleration a(t) = −8 m/s². Starting at 30 m/s, its velocity is v(t) = 30 − 8t. How far does it travel before stopping? Integrate v(t) from t=0 to when v=0.
2. **Solar panel output:** A panel produces power P(t) = 500·sin(πt/12) watts for t = 0 to 12 hours (sunrise to sunset). How many watt-hours of energy are generated in a day?
3. **Your own idea:** Think of any quantity that changes over time — temperature, heart rate, speed — write its rate function and integrate to find the total!